# 09. Uncertainty: prediction intervals and exceedance probabilities

Method chosen with the project owner (task 4.3.1): **split-conformal intervals** around the headline model (LightGBM, `forecast` feature set) and **Platt-scaled probabilities** of a daily mean at or above 55.5 ug/m3 (35.5 as a secondary alert level).

How it works, per walk-forward block: the last 365 training target days are the calibration year (with a gap equal to the lead). A copy of the model fitted on the earlier rows forecasts the calibration year, and its errors set the interval widths and the probability curve. The model used for the test days is refitted on all training rows. Two interval types are compared: `plain` (fixed error quantiles) and `scaled` (error quantiles relative to the forecast level, so polluted days get wider intervals). Intervals are asymmetric because the errors are skewed. Probabilities come from logistic regression on the forecast (`platt`), from the calibration errors directly (`conformal`), and a reference `base` that always gives the calibration-year base rate.

Why this suits the data: about 1,800 daily rows, so flexible quantile models would be unstable; no distribution is assumed; and the coverage can be checked with the same walk-forward protocol. Why it can fail: coverage is guaranteed only on average if calibration and test days are exchangeable, but PM2.5 errors come in spells (notebook 07b), and the calibration year uses observed weather where the real-forecast test uses archived forecasts.

The first two blocks (winter 2020-21 and March to October 2021) are skipped because the calibration model would have too little history. Everything below is therefore for winter 2021-22 onwards.

In [1]:
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from aeron import uncertainty
from aeron.evaluation import metrics, splits
from aeron.models import trees

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
daily = pd.read_csv(
    ROOT / "data" / "processed" / "daily_us_embassy.csv", index_col=0, parse_dates=True
)
blocks = splits.seasonal_blocks(2020, date(2025, 2, 25))
LEADS, VIEWS = (1, 2, 3), ("observed", "forecast")


def make():
    return trees.lightgbm_model("forecast", "lgbm_forecast_weather")


res = {}
for lead in LEADS:
    out = uncertainty.run_intervals(daily, make, lead, blocks, VIEWS)
    for view, frame in out.items():
        res[(view, lead)] = frame
print({k: len(v) for k, v in res.items()})
print("blocks used:", list(res[("observed", 1)]["block"].unique()))

{('observed', 1): 908, ('forecast', 1): 363, ('observed', 2): 885, ('forecast', 2): 358, ('observed', 3): 873, ('forecast', 3): 354}
blocks used: ['winter 2021-22', 'mar-oct 2022', 'winter 2022-23', 'mar-oct 2023', 'winter 2023-24', 'mar-oct 2024', 'winter 2024-25']


## 4.3.2 Interval coverage

Share of days whose observed daily mean falls inside the interval (nominal 90% and 80%), with 95% block-bootstrap intervals (14-day blocks). Plain against level-scaled intervals.

In [2]:
def group(frame, g):
    return frame if g == "all" else frame[frame["season"] == g]


rows = []
for (view, lead), f in res.items():
    for g in ("winter", "other", "all"):
        sub = group(f, g)
        for level in (90, 80):
            for kind in ("plain", "scaled"):
                lo, hi = f"lo{level}_{kind}", f"hi{level}_{kind}"
                c = uncertainty.coverage_interval(sub, lo, hi)
                rows.append(
                    {
                        "view": view,
                        "lead": lead,
                        "group": g,
                        "nominal": level,
                        "type": kind,
                        "n": c["n"],
                        "coverage": round(c["coverage"], 3),
                        "ci": f"[{c['lo']:.2f}, {c['hi']:.2f}]",
                        "mean width": round(uncertainty.mean_width(sub, lo, hi), 1),
                    }
                )
cov = pd.DataFrame(rows)
cov[cov["nominal"] == 90].drop(columns="nominal").set_index(
    ["view", "lead", "group", "type"]
)

n  coverage            ci  mean width
view     lead group  type                                           
observed 1    winter plain   379     0.786  [0.73, 0.84]        48.4
                     scaled  379     0.855  [0.80, 0.90]        60.7
              other  plain   529     0.949  [0.92, 0.97]        49.2
                     scaled  529     0.849  [0.81, 0.88]        26.3
              all    plain   908     0.881  [0.85, 0.91]        48.9
                     scaled  908     0.851  [0.82, 0.88]        40.7
forecast 1    winter plain   146     0.719  [0.66, 0.78]        61.7
                     scaled  146     0.925  [0.89, 0.96]        86.7
              other  plain   217     0.825  [0.72, 0.90]        52.7
                     scaled  217     0.742  [0.64, 0.82]        34.9
              all    plain   363     0.782  [0.72, 0.84]        56.4
                     scaled  363     0.815  [0.75, 0.87]        55.8
observed 2    winter plain   369     0.751  [0.68, 0.81]        55.5
                     scaled  369     0.846  [0.78, 0.91]        69.3
              other  plain   516     0.955  [0.92, 0.98]        67.0
                     scaled  516     0.864  [0.81, 0.91]        37.8
              all    plain   885     0.870  [0.83, 0.91]        62.2
                     scaled  885     0.856  [0.82, 0.89]        50.9
forecast 2    winter plain   143     0.699  [0.63, 0.76]        74.6
                     scaled  143     0.874  [0.81, 0.93]       115.3
              other  plain   215     0.842  [0.73, 0.93]        64.2
                     scaled  215     0.735  [0.62, 0.84]        50.6
              all    plain   358     0.785  [0.72, 0.85]        68.4
                     scaled  358     0.791  [0.72, 0.85]        76.4
observed 3    winter plain   361     0.753  [0.67, 0.83]        60.9
                     scaled  361     0.803  [0.73, 0.87]        72.0
              other  plain   512     0.967  [0.94, 0.99]        75.0
                     scaled  512     0.891  [0.85, 0.93]        41.0
              all    plain   873     0.879  [0.84, 0.92]        69.2
                     scaled  873     0.855  [0.81, 0.89]        53.8
forecast 3    winter plain   140     0.679  [0.59, 0.76]        80.8
                     scaled  140     0.886  [0.83, 0.94]       126.6
              other  plain   214     0.850  [0.74, 0.94]        72.8
                     scaled  214     0.743  [0.64, 0.84]        54.8
              all    plain   354     0.782  [0.71, 0.85]        76.0
                     scaled  354     0.799  [0.74, 0.86]        83.2

In [3]:
cov[cov["nominal"] == 80].drop(columns="nominal").set_index(
    ["view", "lead", "group", "type"]
)

n  coverage            ci  mean width
view     lead group  type                                           
observed 1    winter plain   379     0.612  [0.54, 0.68]        31.0
                     scaled  379     0.720  [0.66, 0.78]        42.7
              other  plain   529     0.885  [0.85, 0.92]        30.2
                     scaled  529     0.741  [0.69, 0.79]        18.6
              all    plain   908     0.771  [0.72, 0.82]        30.5
                     scaled  908     0.732  [0.69, 0.77]        28.7
forecast 1    winter plain   146     0.596  [0.50, 0.68]        38.0
                     scaled  146     0.815  [0.75, 0.87]        60.8
              other  plain   217     0.682  [0.58, 0.78]        33.6
                     scaled  217     0.530  [0.45, 0.60]        22.9
              all    plain   363     0.647  [0.58, 0.71]        35.4
                     scaled  363     0.645  [0.57, 0.72]        38.1
observed 2    winter plain   369     0.610  [0.54, 0.68]        35.6
                     scaled  369     0.707  [0.63, 0.78]        48.7
              other  plain   516     0.868  [0.81, 0.91]        39.0
                     scaled  516     0.738  [0.67, 0.80]        25.7
              all    plain   885     0.760  [0.71, 0.81]        37.6
                     scaled  885     0.725  [0.68, 0.77]        35.3
forecast 2    winter plain   143     0.573  [0.48, 0.67]        45.0
                     scaled  143     0.790  [0.71, 0.86]        76.3
              other  plain   215     0.698  [0.57, 0.81]        41.5
                     scaled  215     0.628  [0.53, 0.72]        30.8
              all    plain   358     0.648  [0.57, 0.73]        42.9
                     scaled  358     0.693  [0.63, 0.75]        49.0
observed 3    winter plain   361     0.648  [0.56, 0.73]        40.0
                     scaled  361     0.720  [0.64, 0.80]        53.4
              other  plain   512     0.908  [0.87, 0.94]        43.2
                     scaled  512     0.752  [0.70, 0.80]        28.7
              all    plain   873     0.801  [0.75, 0.85]        41.9
                     scaled  873     0.739  [0.70, 0.78]        38.9
forecast 3    winter plain   140     0.564  [0.49, 0.65]        51.3
                     scaled  140     0.814  [0.75, 0.88]        92.7
              other  plain   214     0.668  [0.53, 0.79]        43.3
                     scaled  214     0.598  [0.50, 0.70]        38.6
              all    plain   354     0.627  [0.54, 0.71]        46.4
                     scaled  354     0.684  [0.61, 0.76]        59.9

### Coverage of the 90% scaled interval by forecast level (winter)

In [4]:
edges, labels = [0, 25, 55.5, 100, np.inf], ["<25", "25-55.5", "55.5-100", ">=100"]
rows = []
for (view, lead), f in res.items():
    w = f[f["season"] == "winter"].copy()
    w["level"] = pd.cut(w["pred"], edges, labels=labels, right=False)
    for lvl, g in w.groupby("level", observed=False):
        if len(g) == 0:
            continue
        for kind in ("plain", "scaled"):
            rows.append(
                {
                    "view": view,
                    "lead": lead,
                    "forecast level": lvl,
                    "type": kind,
                    "days": len(g),
                    "coverage": round(
                        uncertainty.coverage(g, f"lo90_{kind}", f"hi90_{kind}"), 2
                    ),
                    "below": round(float((g["y"] < g[f"lo90_{kind}"]).mean()), 2),
                    "above": round(float((g["y"] > g[f"hi90_{kind}"]).mean()), 2),
                }
            )
pd.DataFrame(rows).set_index(["view", "lead", "forecast level", "type"])

days  coverage  below  above
view     lead forecast level type                                
observed 1    <25            plain     24      0.96   0.04   0.00
                             scaled    24      0.92   0.08   0.00
              25-55.5        plain    200      0.88   0.07   0.04
                             scaled   200      0.82   0.10   0.09
              55.5-100       plain    135      0.67   0.14   0.19
                             scaled   135      0.90   0.01   0.09
              >=100          plain     20      0.40   0.10   0.50
                             scaled    20      0.85   0.00   0.15
forecast 1    <25            plain      6      1.00   0.00   0.00
                             scaled     6      1.00   0.00   0.00
              25-55.5        plain     64      0.81   0.19   0.00
                             scaled    64      0.92   0.06   0.02
              55.5-100       plain     68      0.65   0.34   0.01
                             scaled    68      0.93   0.07   0.00
              >=100          plain      8      0.38   0.50   0.12
                             scaled     8      0.88   0.12   0.00
observed 2    <25            plain     14      1.00   0.00   0.00
                             scaled    14      0.93   0.07   0.00
              25-55.5        plain    226      0.82   0.09   0.09
                             scaled   226      0.81   0.08   0.11
              55.5-100       plain    122      0.60   0.14   0.26
                             scaled   122      0.89   0.01   0.11
              >=100          plain      7      0.71   0.29   0.00
                             scaled     7      1.00   0.00   0.00
forecast 2    <25            plain      3      1.00   0.00   0.00
                             scaled     3      0.67   0.00   0.33
              25-55.5        plain     64      0.81   0.16   0.03
                             scaled    64      0.86   0.08   0.06
              55.5-100       plain     72      0.60   0.36   0.04
                             scaled    72      0.89   0.10   0.01
              >=100          plain      4      0.50   0.50   0.00
                             scaled     4      1.00   0.00   0.00
observed 3    <25            plain     10      0.80   0.00   0.20
                             scaled    10      0.70   0.00   0.30
              25-55.5        plain    235      0.80   0.11   0.10
                             scaled   235      0.78   0.08   0.14
              55.5-100       plain    112      0.65   0.12   0.23
                             scaled   112      0.85   0.02   0.13
              >=100          plain      4      1.00   0.00   0.00
                             scaled     4      1.00   0.00   0.00
forecast 3    <25            plain      1      1.00   0.00   0.00
                             scaled     1      1.00   0.00   0.00
              25-55.5        plain     67      0.78   0.19   0.03
                             scaled    67      0.87   0.09   0.04
              55.5-100       plain     60      0.55   0.42   0.03
                             scaled    60      0.88   0.12   0.00
              >=100          plain     12      0.75   0.25   0.00
                             scaled    12      1.00   0.00   0.00

### Coverage of the 90% scaled interval by test block

In [5]:
rows = {}
for lead in LEADS:
    for view in VIEWS:
        f = res[(view, lead)]
        rows[(view, lead)] = {
            b: round(uncertainty.coverage(g, "lo90_scaled", "hi90_scaled"), 2)
            for b, g in f.groupby("block", sort=False)
        }
pd.DataFrame(rows).T

,,winter 2021-22,mar-oct 2022,winter 2022-23,mar-oct 2023,winter 2023-24,mar-oct 2024,winter 2024-25
observed,1,0.85,0.94,0.86,0.86,0.75,0.80,0.94
forecast,1,NaN,NaN,NaN,NaN,0.89,0.74,0.94
observed,2,0.86,0.96,0.75,0.88,0.73,0.80,0.99
forecast,2,NaN,NaN,NaN,NaN,0.76,0.73,0.92
observed,3,0.80,0.98,0.62,0.90,0.74,0.85,0.99
forecast,3,NaN,NaN,NaN,NaN,0.78,0.74,0.92


### Do misses come in spells?

Chance that the observation is outside the 90% scaled interval on a day, and on a day right after a miss (consecutive days only).

In [6]:
rows = []
for (view, lead), f in res.items():
    w = f[f["season"] == "winter"]
    rows.append(
        {
            "view": view,
            "lead": lead,
            **{
                k: round(v, 2) if isinstance(v, float) else v
                for k, v in uncertainty.miss_clustering(
                    w, "lo90_scaled", "hi90_scaled"
                ).items()
            },
        }
    )
pd.DataFrame(rows).set_index(["view", "lead"])

,,miss_rate,miss_rate_after_miss,days_after_miss
view,lead,,,
observed,1,0.15,0.21,53
forecast,1,0.08,0.00,11
observed,2,0.15,0.39,51
forecast,2,0.13,0.06,17
observed,3,0.20,0.44,63
forecast,3,0.11,0.14,14


## Exceedance probabilities

Brier score (lower is better), Brier skill against always forecasting the calibration-year base rate (`base`), expected calibration error (ECE, ten equal-count bins) and AUC, winter days. Threshold 55.5 ug/m3.

In [7]:
def prob_table(thr, group_name="winter"):
    tag = f"{thr:g}"
    rows = []
    for (view, lead), f in res.items():
        sub = group(f, group_name)
        for kind in ("platt", "conformal", "base"):
            m = metrics.probability_metrics(sub["y"], sub[f"p{tag}_{kind}"], thr)
            rows.append(
                {
                    "view": view,
                    "lead": lead,
                    "method": kind,
                    "n": m["n"],
                    "brier": round(m["brier"], 3),
                    "skill vs base rate": round(m["brier_skill"], 3),
                    "ece": round(m["ece"], 3),
                    "auc": round(m["auc"], 3),
                    "mean p": round(float(sub[f"p{tag}_{kind}"].mean()), 2),
                    "observed rate": round(float((sub["y"] >= thr).mean()), 2),
                }
            )
    return pd.DataFrame(rows).set_index(["view", "lead", "method"])


prob_table(55.5)

n  brier  skill vs base rate    ece    auc  mean p  observed rate
view     lead method                                                                        
observed 1    platt      379  0.133               0.462  0.053  0.887    0.43           0.45
              conformal  379  0.133               0.464  0.080  0.899    0.37           0.45
              base       379  0.325              -0.313  0.301  0.411    0.19           0.45
forecast 1    platt      146  0.197               0.206  0.168  0.839    0.59           0.45
              conformal  146  0.159               0.357  0.067  0.848    0.44           0.45
              base       146  0.318              -0.283  0.267  0.530    0.19           0.45
observed 2    platt      369  0.175               0.295  0.107  0.838    0.40           0.46
              conformal  369  0.175               0.294  0.129  0.854    0.33           0.46
              base       369  0.326              -0.315  0.303  0.416    0.19           0.46
forecast 2    platt      143  0.233               0.061  0.187  0.776    0.60           0.45
              conformal  143  0.190               0.235  0.066  0.784    0.42           0.45
              base       143  0.320              -0.291  0.271  0.540    0.18           0.45
observed 3    platt      361  0.181               0.269  0.094  0.821    0.39           0.45
              conformal  361  0.182               0.266  0.133  0.839    0.32           0.45
              base       361  0.321              -0.299  0.296  0.422    0.19           0.45
forecast 3    platt      140  0.244               0.015  0.200  0.761    0.63           0.46
              conformal  140  0.194               0.219  0.096  0.769    0.45           0.46
              base       140  0.321              -0.294  0.274  0.550    0.18           0.46

In [8]:
prob_table(35.5)

n  brier  skill vs base rate    ece    auc  mean p  observed rate
view     lead method                                                                        
observed 1    platt      379  0.110               0.468  0.049  0.911    0.70           0.71
              conformal  379  0.112               0.456  0.047  0.912    0.66           0.71
              base       379  0.343              -0.665  0.356  0.413    0.37           0.71
forecast 1    platt      146  0.135               0.300  0.098  0.826    0.81           0.74
              conformal  146  0.132               0.314  0.081  0.823    0.72           0.74
              base       146  0.382              -0.984  0.437  0.555    0.30           0.74
observed 2    platt      369  0.132               0.366  0.063  0.875    0.67           0.70
              conformal  369  0.132               0.367  0.071  0.877    0.64           0.70
              base       369  0.342              -0.641  0.348  0.414    0.37           0.70
forecast 2    platt      143  0.168               0.123  0.120  0.758    0.82           0.74
              conformal  143  0.161               0.160  0.094  0.751    0.71           0.74
              base       143  0.383              -0.997  0.440  0.562    0.30           0.74
observed 3    platt      361  0.143               0.320  0.071  0.865    0.64           0.70
              conformal  361  0.143               0.320  0.071  0.863    0.63           0.70
              base       361  0.338              -0.614  0.348  0.424    0.37           0.70
forecast 3    platt      140  0.176               0.095  0.133  0.768    0.83           0.74
              conformal  140  0.160               0.176  0.086  0.759    0.72           0.74
              base       140  0.380              -0.954  0.434  0.582    0.30           0.74

Paired Brier difference, Platt minus conformal probability (negative favours Platt), winter, 55.5 ug/m3, with block-bootstrap intervals.

In [9]:
rows = []
for (view, lead), f in res.items():
    w = f[f["season"] == "winter"].dropna(subset=["y"])
    ev = (w["y"] >= 55.5).astype(float)
    a, b = (w["p55.5_platt"] - ev) ** 2, (w["p55.5_conformal"] - ev) ** 2
    d = metrics.paired_block_bootstrap(w["target_date"], a, b)
    rows.append(
        {
            "view": view,
            "lead": lead,
            "difference": f"{d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]",
        }
    )
pd.DataFrame(rows).set_index(["view", "lead"])

,,difference
view,lead,
observed,1,"+0.0006 [-0.0081, +0.0095]"
forecast,1,"+0.0374 [+0.0117, +0.0614]"
observed,2,"-0.0003 [-0.0150, +0.0131]"
forecast,2,"+0.0430 [+0.0045, +0.0791]"
observed,3,"-0.0010 [-0.0161, +0.0122]"
forecast,3,"+0.0504 [+0.0131, +0.0901]"


### Reliability of the Platt probability (winter, 55.5 ug/m3, five equal-count bins)

In [10]:
for view in VIEWS:
    for lead in (1, 3):
        w = res[(view, lead)]
        w = w[w["season"] == "winter"]
        t = metrics.reliability_table(
            (w["y"] >= 55.5).astype(float), w["p55.5_platt"], 5
        )
        print(view, "lead", lead)
        print(t.round(2).to_string(index=False))
        print()

observed lead 1
 mean_predicted  observed  n
           0.02      0.11 76
           0.09      0.11 76
           0.32      0.39 76
           0.76      0.74 76
           0.98      0.93 75

observed lead 3
 mean_predicted  observed  n
           0.03      0.10 73
           0.10      0.25 72
           0.25      0.40 72
           0.63      0.62 72
           0.96      0.88 72

forecast lead 1
 mean_predicted  observed  n
           0.05      0.10 30
           0.29      0.14 29
           0.70      0.55 29
           0.96      0.62 29
           0.99      0.86 29

forecast lead 3
 mean_predicted  observed  n
           0.12      0.18 28
           0.37      0.25 28
           0.71      0.39 28
           0.95      0.71 28
           1.00      0.75 28



### Using the probability as an alert: precision and recall when the probability is at least 0.5, against the hard rule 'forecast at or above 55.5' (winter)

In [11]:
rows = []
for (view, lead), f in res.items():
    w = f[f["season"] == "winter"]
    truth = w["y"] >= 55.5
    for label, flag in (
        ("forecast >= 55.5", w["pred"] >= 55.5),
        ("Platt p >= 0.5", w["p55.5_platt"] >= 0.5),
    ):
        tp = float((truth & flag).sum())
        fp = float((~truth & flag).sum())
        fn = float((truth & ~flag).sum())
        rows.append(
            {
                "view": view,
                "lead": lead,
                "rule": label,
                "precision": round(tp / (tp + fp), 2),
                "recall": round(tp / (tp + fn), 2),
                "f1": round(2 * tp / (2 * tp + fp + fn), 2),
            }
        )
pd.DataFrame(rows).set_index(["view", "lead", "rule"])

precision  recall    f1
view     lead rule                                     
observed 1    forecast >= 55.5       0.83    0.75  0.79
              Platt p >= 0.5         0.83    0.76  0.79
forecast 1    forecast >= 55.5       0.71    0.82  0.76
              Platt p >= 0.5         0.67    0.89  0.77
observed 2    forecast >= 55.5       0.81    0.62  0.70
              Platt p >= 0.5         0.78    0.64  0.71
forecast 2    forecast >= 55.5       0.64    0.75  0.70
              Platt p >= 0.5         0.60    0.80  0.68
observed 3    forecast >= 55.5       0.82    0.59  0.68
              Platt p >= 0.5         0.77    0.60  0.67
forecast 3    forecast >= 55.5       0.68    0.77  0.72
              Platt p >= 0.5         0.60    0.83  0.70

## Saved tables

In [12]:
target = ROOT / "reports" / "model_evaluation"
cov.to_csv(target / "interval_coverage.csv", index=False)
pt = pd.concat(
    [prob_table(55.5).assign(threshold=55.5), prob_table(35.5).assign(threshold=35.5)]
)
pt.reset_index().to_csv(target / "probability_metrics.csv", index=False)
print("saved interval_coverage.csv and probability_metrics.csv")

saved interval_coverage.csv and probability_metrics.csv
